In [8]:
import requests

def get_lol_match_history(summoner_name, tagline="NA1", platform="na1", api_key="YOUR_API_KEY"):
    """
    Fetch match history for a League of Legends player (Match v5 API)

    Args:
        summoner_name: Player's Riot ID name (without #tag)
        tagline: Riot ID tagline (e.g. "NA1")
        platform: Platform region (na1, euw1, kr, etc.)
        api_key: Riot Games API key

    Returns:
        List of match IDs
    """
    # Map platform to regional cluster for match v5
    regional_map = {
        "na1": "americas", "br1": "americas", "la1": "americas", "la2": "americas",
        "euw1": "europe", "eun1": "europe", "tr1": "europe", "ru": "europe",
        "kr": "asia", "jp1": "asia",
        "oc1": "sea", "ph2": "sea", "sg2": "sea", "th2": "sea", "tw2": "sea", "vn2": "sea"
    }
    region = regional_map.get(platform, "americas")
    headers = {"X-Riot-Token": api_key}

    # Step 1: Get PUUID via Riot Account API
    account_url = f"https://{region}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{summoner_name}/{tagline}"
    account_resp = requests.get(account_url, headers=headers)
    account_resp.raise_for_status()
    puuid = account_resp.json()["puuid"]

    # Step 2: Get match history using PUUID (Match v5)
    matches_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    matches_params = {"start": 0, "count": 50}
    matches_resp = requests.get(matches_url, headers=headers, params=matches_params)
    matches_resp.raise_for_status()

    return matches_resp.json()


# Usage — note: pass tagline separately, and use header-based auth
match_history = get_lol_match_history(
    summoner_name="sacredswords15",
    tagline="NA1",
    platform="na1",
    api_key="RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40".strip()  # Regenerate at developer.riotgames.com
)

print(match_history)

['NA1_5505092770', 'NA1_5505066777', 'NA1_5505047083', 'NA1_5505026084', 'NA1_5505007074', 'NA1_5504989622', 'NA1_5503949838', 'NA1_5502022714', 'NA1_5500410199', 'NA1_5500390042', 'NA1_5500362923', 'NA1_5499946088', 'NA1_5499926127', 'NA1_5499909510', 'NA1_5499891527', 'NA1_5499594693', 'NA1_5499299146', 'NA1_5499272747', 'NA1_5499240552', 'NA1_5499219783', 'NA1_5498904360', 'NA1_5498873097', 'NA1_5498844454', 'NA1_5498795087', 'NA1_5498747088', 'NA1_5498703911', 'NA1_5498650002', 'NA1_5497374011', 'NA1_5497354466', 'NA1_5497336108', 'NA1_5496580631', 'NA1_5496564432', 'NA1_5496545993', 'NA1_5496542872', 'NA1_5496274223', 'NA1_5496127761', 'NA1_5495787736', 'NA1_5495773022', 'NA1_5495759135', 'NA1_5495749767', 'NA1_5494992804', 'NA1_5494986216', 'NA1_5494827406', 'NA1_5494791740', 'NA1_5494498941', 'NA1_5494481895', 'NA1_5494457075', 'NA1_5494436980', 'NA1_5494413961', 'NA1_5494369024']


In [6]:
import requests

api_key = "RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40"
headers = {"X-Riot-Token": api_key}
url = "https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/sacredswords15/NA1"

r = requests.get(url, headers=headers)
print(r.status_code)
print(r.json())

200
{'puuid': 'QUHKsuLP3VqPLczikq2CI3rYU3s0FlBkmCEDzkPjN46St8z1m0Cm0g40ehp6WQjF12MxgrLY1x9Lrw', 'gameName': 'sacredswords15', 'tagLine': 'NA1'}


In [9]:
def is_ranked_solo_duo(match_id, region="americas"):
    """
    Check if a match is a ranked solo/duo game
    
    Args:
        match_id: The match ID to check
        region: Regional cluster (default: "americas")
    
    Returns:
        Boolean indicating if the match is ranked solo/duo
    """
    match_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    match_resp = requests.get(match_url, headers=headers)
    match_resp.raise_for_status()
    
    match_data = match_resp.json()
    queue_id = match_data["info"]["queueId"]
    
    # Queue ID 420 = Ranked Solo/Duo
    return queue_id == 420

# Test with the first match
first_match = match_history[0]
result = is_ranked_solo_duo(first_match)
print(f"{first_match} is ranked solo/duo: {result}")

NA1_5505092770 is ranked solo/duo: True
